Thato Phenyo Molete
BLCH9X2 Assignment 3: Transaction Engine Lab
SimpleBlockchain payment layer.

In [1]:
from __future__ import annotations
 
import json
import time
from typing import Any, Optional
 
 
class Transaction:
    """Minimal payment message for SimpleBlockchain.
 
    Fields:
        sender (str): payer identity / address
        recipient (str): payee identity / address
        amount (float): value transferred, must be > 0
        timestamp (float): Unix epoch time of creation
        signature (str, None): authorisation placeholder until wallets
            and ECDSA signing are introduced (Lecture 04)
    """
 
    def __init__(
        self,
        sender: str,
        recipient: str,
        amount: float,
        timestamp: Optional[float] = None,
        signature: Optional[str] = None,
    ) -> None:
        self.sender = sender
        self.recipient = recipient
        self.amount = amount
        # Default to "now" (Unix time) when the caller omits timestamp
        self.timestamp = time.time() if timestamp is None else float(timestamp)
        self.signature = signature  # placeholder until Lecture 04
 
    def to_dict(self) -> dict[str, Any]:
        """Return a JSON-serialisable dictionary of all five fields."""
        return {
            "sender": self.sender,
            "recipient": self.recipient,
            "amount": self.amount,
            "timestamp": self.timestamp,
            "signature": self.signature,
        }
 
    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "Transaction":
        """Construct a Transaction from a dictionary."""
        return cls(
            sender=data["sender"],
            recipient=data["recipient"],
            amount=data["amount"],
            timestamp=data.get("timestamp"),
            signature=data.get("signature"),
        )
 
    def to_json(self) -> str:
        """Serialise to a canonical JSON string.
 
        sort_keys=True and fixed separators give a deterministic byte
        representation - required later for hashing/tx-id derivation.
        """
        return json.dumps(self.to_dict(), sort_keys=True, separators=(",", ":"))
 
    @classmethod
    def from_json(cls, payload: str) -> "Transaction":
        """Deserialise from a JSON string."""
        return cls.from_dict(json.loads(payload))
 
    def __repr__(self) -> str:
        return (
            f"Transaction({self.sender!r} -> {self.recipient!r}, "
            f"amount={self.amount}, signature={self.signature!r})"
        )
 
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Transaction):
            return NotImplemented
        return self.to_dict() == other.to_dict()
 
 
def validate_transaction(tx: Transaction) -> bool:
    """Structural validation only (no signature or balance checks).
 
    Rejects a transaction if:
      1. sender or recipient is missing / empty / whitespace-only
      2. amount is not numeric, or amount <= 0
         (bool is technically a subclass of int in Python - explicitly
         reject True/False masquerading as an amount)
      3. timestamp is missing or cannot be interpreted as a number
    """
    if not isinstance(tx.sender, str) or not tx.sender.strip():
        return False
    if not isinstance(tx.recipient, str) or not tx.recipient.strip():
        return False
    if not isinstance(tx.amount, (int, float)) or isinstance(tx.amount, bool):
        return False
    if tx.amount <= 0:
        return False
    if tx.timestamp is None:
        return False
    try:
        float(tx.timestamp)
    except (TypeError, ValueError):
        return False
    return True
 
 
class Mempool:
    """Local waiting room for structurally valid transactions.
 
    Not global consensus, and not the ledger - just a node's own pool of
    candidate transactions waiting to be picked up by a miner.
    """
 
    def __init__(self) -> None:
        self._txs: list[Transaction] = []
 
    def add(self, tx: Transaction) -> bool:
        """Validate and, if valid, append. Return True iff accepted."""
        if not validate_transaction(tx):
            return False
        self._txs.append(tx)
        return True
 
    def get_all(self) -> list[Transaction]:
        """Return a shallow copy of pending transactions (callers cannot
        mutate the internal list through this)."""
        return list(self._txs)
 
    def __len__(self) -> int:
        return len(self._txs)
 
    def clear(self) -> None:
        self._txs.clear()
 
 
if __name__ == "__main__":
    # Quick manual sanity check when running this file directly.
    tx = Transaction("Phenyo", "Thato", 10.0)
    print(tx)
    print(tx.to_json())
    print("Valid?", validate_transaction(tx))

Transaction('Phenyo' -> 'Thato', amount=10.0, signature=None)
{"amount":10.0,"recipient":"Thato","sender":"Phenyo","signature":null,"timestamp":1787393924.04848}
Valid? True
